<a href="https://colab.research.google.com/github/mixxr/gcolab-ml/blob/main/funnel/01_the_funnel_first_molecule.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Episode 1 — The Funnel & Your First Molecule
**Drug Discovery in Code** · [Life Sciences AI Applications]
There are ~10⁶⁰ possible drug-like molecules. Drug discovery is the search for the few that
could become a medicine — and it works like a **funnel**: start with a huge library, then narrow,
stage by stage, spending cheap computation to avoid expensive lab work.
In this first notebook you'll set up RDKit, load a molecule from text, draw it, and measure the
properties that decide whether it can be a drug.
> Educational only — not medical advice.

## The funnel
```
  millions of molecules
        │  drug-like filter        (Ep 4)
        │  similar to known actives (Ep 5)
        │  activity model says 'yes' (Ep 6)
        │  docks into the target    (Ep 7-8)
        ▼
  a handful worth making
```
Two things anchor everything: a **target** (a protein) and, ideally, some **known actives**.
Our running target for the series is **EGFR**. Let's start with the basics.

## Setup
Install once (see the repo README), then import RDKit — the open-source cheminformatics toolkit.

In [2]:
! wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
! bash Miniconda3-latest-Linux-x86_64.sh -b -f -p /usr/local

--2026-10-09 06:41:13--  https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 104.16.32.241, 104.16.191.158, 2606:4700::6810:20f1, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|104.16.32.241|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 197973354 (189M) [application/octet-stream]
Saving to: ‘Miniconda3-latest-Linux-x86_64.sh’

Miniconda3-latest-L 100%[===================>] 188.80M   267MB/s    in 0.7s    

2026-10-09 06:41:14 (267 MB/s) - ‘Miniconda3-latest-Linux-x86_64.sh’ saved [197973354/197973354]

PREFIX=/usr/local
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please 

In [3]:
import sys
!{sys.executable} -m pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 19.2 MB/s eta 0:00:00


In [4]:
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors, Lipinski
import pandas as pd

## Your first molecule — from a SMILES string
**SMILES** is a compact text notation for any molecule. RDKit parses it into a molecule object
we can draw and analyze. Here's aspirin:

In [ ]:
aspirin = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O")
Draw.MolToImage(aspirin, size=(360, 300))

## Measuring a molecule
Molecules have properties that predict whether they can be oral drugs. Four key ones:

In [ ]:
print("Molecular weight:", round(Descriptors.MolWt(aspirin), 1))
print("LogP (greasiness):", round(Descriptors.MolLogP(aspirin), 2))
print("H-bond donors:", Lipinski.NumHDonors(aspirin))
print("H-bond acceptors:", Lipinski.NumHAcceptors(aspirin))

## Lipinski's Rule of Five
The most famous rule of thumb in the field: oral drugs usually have **MW ≤ 500**, **LogP ≤ 5**,
**≤ 5 H-bond donors**, and **≤ 10 H-bond acceptors**. Let's turn it into a function and score a
few molecules — including **erlotinib**, a real EGFR inhibitor, and **reserpine**, which breaks
the rules.

In [ ]:
def passes_ro5(mol):
    return (Descriptors.MolWt(mol) <= 500
            and Descriptors.MolLogP(mol) <= 5
            and Lipinski.NumHDonors(mol) <= 5
            and Lipinski.NumHAcceptors(mol) <= 10)

molecules = {
    "Aspirin":               "CC(=O)Oc1ccccc1C(=O)O",
    "Caffeine":              "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "Erlotinib (EGFR drug)": "COCCOc1cc2ncnc(Nc3cccc(C#C)c3)c2cc1OCCOC",
    "Reserpine":             "COC(=O)C1C(OC)C(OC(=O)c2cc(OC)c(OC)c(OC)c2)CC2C1CC1c3[nH]c4cc(OC)ccc4c3CCN1C2",
}

rows = []
for name, smi in molecules.items():
    m = Chem.MolFromSmiles(smi)
    rows.append({
        "Molecule": name,
        "MolWt": round(Descriptors.MolWt(m), 1),
        "LogP": round(Descriptors.MolLogP(m), 2),
        "HBD": Lipinski.NumHDonors(m),
        "HBA": Lipinski.NumHAcceptors(m),
        "Passes Ro5": "Yes" if passes_ro5(m) else "No",
    })

pd.DataFrame(rows)

Reserpine fails on molecular weight — a good reminder that the Rule of Five is a guide, not a law
(plenty of real drugs break it). We'll use these same properties to filter a whole library in **Episode 4**.

In [ ]:
# See all four structures at once
mols = [Chem.MolFromSmiles(s) for s in molecules.values()]
Draw.MolsToGridImage(mols, legends=list(molecules.keys()), molsPerRow=2, subImgSize=(340, 260))

## Recap
- Drug discovery is a **funnel**: narrow millions of molecules to a handful worth making.
- You start from a **target** and **known actives**; the whole workflow runs on free Python tools.
- **RDKit** loads molecules from **SMILES**, draws them, and computes drug-like properties.
- **Lipinski's Rule of Five** is the classic drug-likeness filter.

**Next — Episode 2:** how computers really *represent* molecules (graphs, descriptors & fingerprints) —
the foundation every later step depends on.